In [48]:
import numpy as np
from  PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
from torchvision import datasets, transforms



In [49]:
train_path = "imagewoof_train/train"
val_path = "imagewoof_val/val"

In [50]:
MEAN = (0.5, 0.5, 0.5)
STD  = (0.5, 0.5, 0.5)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(128, scale=(0.35, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

In [51]:
train_dataset = datasets.ImageFolder(root=train_path, transform=train_transform)
val_dataset   = datasets.ImageFolder(root=val_path,   transform=val_transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle= True, num_workers= 2)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle= True, num_workers= 2)

In [52]:
#checking the data sape and the size
image, label = train_dataset[0]
image 
# image.size() #torch.Size([3, 128, 128])

tensor([[[-0.9137, -0.9059, -0.9059,  ..., -0.5137, -0.5216, -0.5294],
         [-0.9216, -0.9137, -0.9137,  ..., -0.5373, -0.5294, -0.5059],
         [-0.9216, -0.9216, -0.9216,  ..., -0.5608, -0.5529, -0.5294],
         ...,
         [-0.4039, -0.3647, -0.3020,  ..., -0.0118, -0.0353, -0.0510],
         [-0.3255, -0.3098, -0.2549,  ..., -0.0275, -0.0431, -0.0588],
         [-0.2392, -0.2471, -0.1922,  ..., -0.0353, -0.0431, -0.0510]],

        [[-0.9137, -0.9059, -0.9059,  ..., -0.4118, -0.4118, -0.4039],
         [-0.9216, -0.9137, -0.9137,  ..., -0.4431, -0.4275, -0.3961],
         [-0.9216, -0.9216, -0.9216,  ..., -0.4824, -0.4510, -0.4196],
         ...,
         [-0.4431, -0.4118, -0.3490,  ..., -0.2863, -0.3020, -0.3176],
         [-0.3647, -0.3490, -0.2941,  ..., -0.3020, -0.3098, -0.3255],
         [-0.2784, -0.2863, -0.2314,  ..., -0.3020, -0.3098, -0.3176]],

        [[-0.9294, -0.9216, -0.9216,  ..., -0.4667, -0.4588, -0.4510],
         [-0.9373, -0.9294, -0.9294,  ..., -0

In [53]:
#Loads the orderd class name from classes.txt
with open("classes.txt") as f:
    class_name = [line.strip() for line in f if line.strip()]

print(class_name)

['0 shih_tzu', '1 rhodesian_ridgeback', '2 beagle', '3 english_foxhound', '4 border_terrier', '5 australian_terrier', '6 golden_retriever', '7 old_english_sheepdog', '8 samoyed', '9 dingo']


In [54]:
import torch.nn as nn

class NeuralNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),   # conv1: [3,128,128] -> [32,128,128]
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),  # conv2
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),                              # -> [32,64,64]

            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # -> [64,64,64]
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),                              # -> [64,32,32]

            nn.AdaptiveAvgPool2d(1),                      # global average pooling -> [64,1,1]
            nn.Flatten(),                                 # -> [64]
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),                   # -> [10] class scores
        )

    def forward(self, x):
        return self.layers(x)

In [55]:
model = NeuralNet()
x = image.unsqueeze(0)      # [1, 3, 128, 128]
out = model(x)
print(out.shape)            # torch.Size([1, 10])

torch.Size([1, 10])


In [56]:
net = NeuralNet()
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [ ]:
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = net.to(device)
print("Training on:", device)

num_epochs = 100
optimizer = torch.optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

best_acc = 0.0
start_time = time.time()

for epoch in range(num_epochs):
    epoch_start = time.time()

    # ---- training ----
    net.train()
    running_loss = 0.0
    correct, total = 0, 0

    for i, data in enumerate(train_loader):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # ---- validation ----
    net.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = net(inputs)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)
    val_acc = val_correct / val_total

    # ---- learning-rate schedule ----
    current_lr = optimizer.param_groups[0]["lr"]
    scheduler.step()

    # ---- timer ----
    epoch_time = time.time() - epoch_start
    elapsed = time.time() - start_time
    remaining = elapsed / (epoch + 1) * (num_epochs - epoch - 1)

    print(f"Epoch {epoch+1}/{num_epochs} | loss {train_loss:.3f} | train acc {train_acc:.3f} | "
          f"val acc {val_acc:.3f} | lr {current_lr:.4f} | "
          f"{epoch_time:.0f}s | elapsed {elapsed/60:.1f} min | ~{remaining/60:.0f} min left")

    # ---- save the best model ----
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(net.state_dict(), "model.pth")
        print("  saved new best model")

total_time = time.time() - start_time
print(f"Done in {total_time/60:.1f} min. Best val accuracy: {best_acc:.3f}")

Epoch 1: loss 2.204 | train acc 0.179 | val acc 0.234
  saved new best model
Epoch 2: loss 2.129 | train acc 0.208 | val acc 0.232
Epoch 3: loss 2.106 | train acc 0.223 | val acc 0.181
Epoch 4: loss 2.090 | train acc 0.227 | val acc 0.195
Epoch 5: loss 2.071 | train acc 0.238 | val acc 0.245
  saved new best model
Epoch 6: loss 2.063 | train acc 0.245 | val acc 0.269
  saved new best model
Epoch 7: loss 2.050 | train acc 0.258 | val acc 0.264
Epoch 8: loss 2.033 | train acc 0.255 | val acc 0.258
Epoch 9: loss 2.028 | train acc 0.267 | val acc 0.259


KeyboardInterrupt: 